In [ ]:
import numpy as np
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import time
from numba import cuda

BLOCK_SIZE = 16

img = mpimg.imread('original_image.jpg')
height, width, channels = img.shape
grid_size = ((width + BLOCK_SIZE - 1) // BLOCK_SIZE,
             (height + BLOCK_SIZE - 1) // BLOCK_SIZE)
block_size = (BLOCK_SIZE, BLOCK_SIZE)
print("Image shape:", img.shape)

In [ ]:
@cuda.jit
def RGB2HSV(src, H, S, V):
    x, y = cuda.grid(2)
    if x < src.shape[1] and y < src.shape[0]:
        r = src[y, x, 0] / 255
        g = src[y, x, 1] / 255
        b = src[y, x, 2] / 255
        mx = max(r, g, b)
        mn = min(r, g, b)
        d = mx - mn

        h = 0.0
        if d > 0:
            if mx == r:
                h = 60 * ((g - b) / d)
            elif mx == g:
                h = 60 * ((b - r) / d + 2)
            else:
                h = 60 * ((r - g) / d + 4)
        if h < 0:
            h += 360

        s = 0.0
        if mx > 0:
            s = d / mx

        H[y, x] = h
        S[y, x] = s
        V[y, x] = mx


@cuda.jit
def HSV2RGB(H, S, V, dst):
    x, y = cuda.grid(2)
    if x < dst.shape[1] and y < dst.shape[0]:
        h = H[y, x]
        s = S[y, x]
        v = V[y, x]
        d = h / 60
        hi = int(d) % 6
        f = d - int(d)
        l = v * (1 - s)
        m = v * (1 - f * s)
        n = v * (1 - (1 - f) * s)

        if hi == 0:
            r, g, b = v, n, l
        elif hi == 1:
            r, g, b = m, v, l
        elif hi == 2:
            r, g, b = l, v, n
        elif hi == 3:
            r, g, b = l, m, v
        elif hi == 4:
            r, g, b = n, l, v
        else:
            r, g, b = v, l, m

        # back to one array (AoS)
        dst[y, x, 0] = r * 255 + 0.5
        dst[y, x, 1] = g * 255 + 0.5
        dst[y, x, 2] = b * 255 + 0.5

In [ ]:
start = time.time()
d_src = cuda.to_device(img)
d_H = cuda.device_array((height, width), np.float32)
d_S = cuda.device_array((height, width), np.float32)
d_V = cuda.device_array((height, width), np.float32)
d_dst = cuda.device_array_like(img)

RGB2HSV[grid_size, block_size](d_src, d_H, d_S, d_V)
HSV2RGB[grid_size, block_size](d_H, d_S, d_V, d_dst)
cuda.synchronize()

result = d_dst.copy_to_host()
print(f"GPU time: {time.time() - start:.4f}s")

diff = np.abs(result.astype(int) - img.astype(int))
print("Max difference:", diff.max())

plt.figure(figsize=(16, 5))
plt.subplot(1, 4, 1); plt.imshow(img); plt.title("Original")
plt.subplot(1, 4, 2); plt.imshow(d_H.copy_to_host(), cmap='hsv'); plt.title("H")
plt.subplot(1, 4, 3); plt.imshow(d_V.copy_to_host(), cmap='gray'); plt.title("V")
plt.subplot(1, 4, 4); plt.imshow(result); plt.title("HSV -> RGB")
plt.savefig('hsv_output.png', bbox_inches='tight')
plt.show()